# UNIT400-TWIN · M1 & M2 in action

**What the twin is.** A Python re-implementation of the Mimic model that runs Unit 400 (R‑400 hydrogenation reactor). The plant's *real* inputs (feed, H2, preheat, jacket flows) go in; the twin computes what Mimic computes: bed temperatures TI401–406 (top → bottom), jacket and V‑410 temperatures, and the analyser 04AT400 (% unreacted feed, 0 = full conversion).

**How the reaction works in the model** (per bed zone i):
- speed f = (T_i − T_START) / (T_IDEAL − T_START), zero below T_START (100 °C); full design rate at T_IDEAL (160 °C)
- rate = f × catalyst activity 04CAT400 × (1000 / feed) — less feed = longer dwell = more conversion per litre
- heat released raises the zone temperature (HEAT_PER_MOLE) → hotter zone → faster reaction: **positive feedback**. That loop is why the bed either holds the reaction or loses it.

**M1** — replay an operating sequence (your `basis.json`, or the generic `basis.example.json`) and, if you have one, compare with a measured steady state in `reference.json`.
**M2** — a catalyst becomes a swappable profile (activity, light‑off, kinetics law, deactivation).

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'twin'))
import basis as Bz
B, REF = Bz.load(), Bz.reference()
print('basis:', B['name'])
import matplotlib.pyplot as plt
import unit400_twin as T
import catalyst as C
T.NOISE = False          # deterministic runs

class PI:                # 04TIC410 in AUTO (V-410 return 110 C), as on the plant
    def __init__(s, kc, ti, direct=False):
        s.kc, s.ti, s.op, s.e1, s.d = kc, ti, 0.0, None, direct
    def u(s, pv, sp, dt=1.0):
        e = (pv - sp) if s.d else (sp - pv)
        if s.e1 is not None:
            s.op += s.kc * ((e - s.e1) + dt / s.ti * e)
        s.op = max(0.0, min(100.0, s.op)); s.e1 = e
        return s.op

SEQ = B['sequence']   # (minutes, inputs changed, label)
TAGS = ['04TI40%d_PV' % i for i in range(1, 7)] + ['04AT400_ATSIM', '04TIC411_PV', '04TIC410_PV']

def run(profile=None, seq=SEQ, every=30, extra_min=0):
    m = C.CatalystTwin(profile); pi = PI(0.5, 120, direct=True); log, marks = [], []
    def go(minutes):
        for _ in range(int(minutes * 60)):
            m.set('04TIC410_OP', pi.u(m.get('04TIC410_PV'), B['tic410_sp'])); m.step(1.0)
            if int(m.t) % every == 0:
                log.append([m.t / 60] + [m.get(t) for t in TAGS])
    for mins, kw, label in seq:
        for k, v in kw.items(): m.set(k, v)
        marks.append((m.t / 60, label)); go(mins)
    if extra_min: go(extra_min)
    return m, log, marks
print('ready')

## M1 — replay of the operating sequence (≈30 s)

In [ ]:
t0 = time.time(); m1, log, marks = run(); print('run time %.0f s' % (time.time() - t0))
t = [r[0] for r in log]
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for i in range(6):
    ax[0].plot(t, [r[1 + i] for r in log], label='TI40%d' % (i + 1))
ax[0].plot(t, [r[8] for r in log], 'k--', lw=1, label='jacket in (TIC411)')
ax[0].set_ylabel('°C'); ax[0].legend(ncol=4, fontsize=8); ax[0].set_title('R-400 bed, top (TI401) to bottom (TI406)')
ax[1].plot(t, [r[7] for r in log], 'r'); ax[1].set_ylabel('04AT400 unreacted %'); ax[1].set_xlabel('minutes')
for x, lab in marks:
    for a in ax: a.axvline(x, color='grey', lw=0.5)
    ax[1].text(x, 102, lab, rotation=90, fontsize=7, va='bottom')
plt.tight_layout(); plt.show()

In [ ]:
bed = list(range(1, 7))
if REF:
    d = []
    print('%-14s %7s %7s %7s' % ('tag', 'measured', 'twin', 'diff'))
    for k, v in REF.items():
        if k == 'name': continue
        w = m1.get(k); print('%-14s %7.1f %7.1f %+7.1f' % (k, v, w, w - v))
        if k.startswith('04TI40'): d.append(w - v)
    print('bed rms error %.1f C' % ((sum(x * x for x in d) / len(d)) ** 0.5))
plt.figure(figsize=(6, 4))
if REF: plt.plot([REF['04TI40%d_PV' % i] for i in bed], bed, 'o-', label='measured')
plt.plot([m1.get('04TI40%d_PV' % i) for i in bed], bed, 's--', label='twin')
plt.gca().invert_yaxis(); plt.yticks(bed, ['TI40%d' % i for i in bed]); plt.xlabel('°C'); plt.legend(); plt.title('Bed profile at steady state'); plt.show()

**Read it:** with the author's private plant data the twin matched a real reaction run within ~8 °C rms on the bed (2.9 % vs 0.9 % unreacted), peak one zone higher than measured — good enough to compare catalysts *against each other*, not a perfect copy.

## M2 — swap the catalyst, same inputs (≈2.5 min for 5 profiles)

In [ ]:
lib = C.load_library(os.path.join('..', 'twin', 'catalysts.json'))
for p in lib.values(): print('%-20s %-11s %s' % (p.name, p.status, p.note))
res = {}
for name, p in lib.items():
    m, lg, _ = run(p); res[name] = (m, lg)
    print('%-20s AT400 %5.1f %%  hot spot %5.1f C' % (name, m.get('04AT400_ATSIM'), m.hot_spot()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (m, lg) in res.items():
    ax[0].plot([m.get('04TI40%d_PV' % i) for i in bed], bed, 'o-', label=name)
    ax[1].plot([r[0] for r in lg], [r[7] for r in lg], label=name)
ax[0].invert_yaxis(); ax[0].set_yticks(bed); ax[0].set_yticklabels(['TI40%d' % i for i in bed]); ax[0].set_xlabel('°C'); ax[0].set_title('Final bed profile')
ax[1].set_ylabel('04AT400 unreacted %'); ax[1].set_xlabel('minutes'); ax[1].set_title('Conversion during the run')
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Read it:**
- **30 % less active → ~42 % unreacted, not 30 %.** Less reaction → less heat → cooler bed → slower reaction. The positive-feedback loop amplifies the loss.
- **Late light‑off (120 °C)** looks almost the same: the bed never gets hot enough for that catalyst to work fully.
- **Arrhenius, Ea 60** — exponential temperature sensitivity: zone 1 runs away (~250 °C). Same plant settings, different catalyst, unsafe. This is what the test bench is for.

### Deactivation — what the operator sees over hours

In [ ]:
p = lib['deactivating_kd005']
m, lg, _ = run(p, every=120, extra_min=240)
tt = [r[0] / 60 for r in lg]
fig, ax1 = plt.subplots(figsize=(10, 4)); ax2 = ax1.twinx()
ax1.plot(tt, [r[7] for r in lg], 'r', label='04AT400 unreacted %')
ax2.plot(tt, [max(r[1:7]) for r in lg], 'b', label='hot spot °C')
ax1.set_xlabel('hours'); ax1.set_ylabel('unreacted %', color='r'); ax2.set_ylabel('hot spot °C', color='b')
plt.title('Deactivating catalyst at constant plant settings'); plt.show()
print('after %.1f h on stream: activity %.0f %%, unreacted %.1f %%' % (m.tos_h, m.get('04CAT400'), m.get('04AT400_ATSIM')))

**Cue for operators:** unreacted feed creeping up while the hot spot drifts *down* at unchanged settings = catalyst dying, not a control problem.

## Try your own catalyst
Edit the numbers and run. `status='illustrative'` until real data exists.

In [ ]:
mine = C.CatalystProfile('my_test', activity_pct=85, t_start=105, t_ideal=165,
                         kinetics='linear', ea_kj_mol=40, kd_per_h=0.0)
m, lg, _ = run(mine)
print('AT400 %.1f %%  hot spot %.1f C  bed %s' % (m.get('04AT400_ATSIM'), m.hot_spot(),
      ' '.join('%.1f' % m.get('04TI40%d_PV' % i) for i in bed)))